In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate
from tensorflow.keras.models import Model
import numpy as np

In [23]:
Conj_val=np.load('Features/conjunctiva_val_features.npy')
Text_val=np.load('Features/text_val.npy')

In [24]:
Conjunctiva_features = np.load("Features/Conjunctiva_train_features.npy")
Textual_features = np.load("Features/text_train.npy")


In [25]:
Conjunctiva_features = tf.convert_to_tensor(Conjunctiva_features, dtype=tf.float32)
Textual_features = tf.convert_to_tensor(Textual_features, dtype=tf.float32)
Conj_val = tf.convert_to_tensor(Conj_val, dtype=tf.float32)
Text_val = tf.convert_to_tensor(Text_val, dtype=tf.float32)

In [26]:
Conjunctiva_features = tf.expand_dims(Conjunctiva_features, axis=1)
Textual_features = tf.expand_dims(Textual_features, axis=1)
Conj_val = tf.expand_dims(Conj_val, axis=1)
Text_val = tf.expand_dims (Text_val, axis =1)

# Self Attention

In [27]:
def self_attention_block(x, num_heads=4):
    d_model = x.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(x, x)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(x + attn)
    return out

In [28]:
conj_input = Input(shape=(1, Conjunctiva_features.shape[-1]), name="Conjunctiva_Input")
text_input = Input(shape=(1, Textual_features.shape[-1]), name="Text_Input")
conj_self = self_attention_block(conj_input)
text_self = self_attention_block(text_input)

# Cross Attention

In [29]:
def cross_attention_block(query, key_value, num_heads=4):
    d_model = query.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(query, key_value)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out


In [30]:
text_cross_conj = cross_attention_block(text_self, conj_self)

In [31]:
from tensorflow.keras.layers import Flatten

conj_flat = Flatten()(conj_self)
text_flat = Flatten()(text_self)
text_cross_flat = Flatten()(text_cross_conj)

# Concatenate
fusion_vector = Concatenate()([conj_flat, text_flat, text_cross_flat])


In [32]:
x = Dense(512, activation='relu')(fusion_vector)
x = Dropout(0.5)(x)

output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x)  # classification
output_hb = Dense(1, activation='linear', name="Hemoglobin")(x)        # regression


In [33]:
final_model = Model(
    inputs=[conj_input, text_input],
    outputs=[output_class, output_hb]
)

final_model.compile(
    optimizer='adam',
    loss={'Anemia_Class':'binary_crossentropy', 'Hemoglobin':'mse'},
    metrics={'Anemia_Class':'accuracy', 'Hemoglobin':'mae'}
)

final_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Conjunctiva_Input   │ (None, 1, 1024)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Input          │ (None, 1, 32)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 1024)   │  4,198,400 │ Conjunctiva_Inpu… │
│ (MultiHeadAttentio… │                   │            │ Conjunctiva_Inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │      4,224 │ Text_Input[0][0], │
│ (MultiHeadAttentio… │                   │            │ Text_Input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 1, 1024)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 1, 32)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 1, 1024)   │          0 │ Conjunctiva_Inpu… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 1, 32)     │          0 │ Text_Input[0][0], │
│                     │                   │            │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 1024)   │      2,048 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │     67,712 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 1, 32)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 1024)      │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1088)      │          0 │ flatten_3[0][0],  │
│ (Concatenate)       │                   │            │ flatten_4[0][0],

 Total params: 4,831,106 (18.43 MB)

 Trainable params: 4,831,106 (18.43 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
y_train_class = np.load("y_train_class.npy")
y_train_hb = np.load("y_train_hb.npy")

In [35]:
y_val_class = np.load("y_val_class.npy")
y_val_hb = np.load("y_val_hb.npy")

In [36]:
print("Conjunctiva features shape:", Conjunctiva_features.shape)
print("Textual features shape:", Textual_features.shape)
print("y_train_class shape:", y_train_class.shape)
print("y_train_hb shape:", y_train_hb.shape)
print("Conj_val shape:", Conj_val.shape)
print("Text_val shape:", Text_val.shape)
print("y_val_class shape:", y_val_class.shape)
print("y_val_hb shape:", y_val_hb.shape)


Conjunctiva features shape: (496, 1, 1024)
Textual features shape: (496, 1, 32)
y_train_class shape: (496,)
y_train_hb shape: (496,)
Conj_val shape: (105, 1, 1024)
Text_val shape: (105, 1, 32)
y_val_class shape: (105,)
y_val_hb shape: (105,)


In [37]:
# Assuming y_train_class (0/1) and y_train_hb (float)
final_model.fit(
    [Conjunctiva_features, Textual_features],
    [y_train_class, y_train_hb],
    validation_data=([Conj_val, Text_val], [y_val_class, y_val_hb]),
    epochs=30,
    batch_size=16
)


Epoch 1/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 8s 90ms/step - Anemia_Class_accuracy: 0.5423 - Anemia_Class_loss: 0.8811 - Hemoglobin_loss: 14.0784 - Hemoglobin_mae: 2.8494 - loss: 14.9595 - val_Anemia_Class_accuracy: 0.4381 - val_Anemia_Class_loss: 0.6962 - val_Hemoglobin_loss: 5.8327 - val_Hemoglobin_mae: 1.8831 - val_loss: 6.2090
Epoch 2/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - Anemia_Class_accuracy: 0.5262 - Anemia_Class_loss: 1.0058 - Hemoglobin_loss: 7.9143 - Hemoglobin_mae: 2.2810 - loss: 8.9201 - val_Anemia_Class_accuracy: 0.6381 - val_Anemia_Class_loss: 0.6644 - val_Hemoglobin_loss: 5.8470 - val_Hemoglobin_mae: 1.9092 - val_loss: 6.3827
Epoch 3/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - Anemia_Class_accuracy: 0.5786 - Anemia_Class_loss: 0.8645 - Hemoglobin_loss: 7.0053 - Hemoglobin_mae: 2.1636 - loss: 7.8698 - val_Anemia_Class_accuracy: 0.6381 - val_Anemia_Class_loss: 0.7027 - val_Hemoglobin_loss: 7.0202 - val_Hemoglobin_mae: 2.0811 - val_loss: 7.1241
Epoch 4/30
31/31 ━━━━━━━━━━━━

In [39]:
# Load test features
Conj_test = np.load("Features/conjunctiva_test_features.npy")
Text_test = np.load("Features/text_test.npy")

# Convert to tensor
Conj_test = tf.convert_to_tensor(Conj_test, dtype=tf.float32)
Text_test = tf.convert_to_tensor(Text_test, dtype=tf.float32)

# Expand dims for attention layer
Conj_test = tf.expand_dims(Conj_test, axis=1)
Text_test = tf.expand_dims(Text_test, axis=1)

# Load test labels
y_test_class = np.load("y_test_class.npy")
y_test_hb = np.load("y_test_hb.npy")


In [40]:
results = final_model.evaluate(
    [Conj_test, Text_test],
    [y_test_class, y_test_hb],
    batch_size=16
)

print("Test Results:", dict(zip(final_model.metrics_names, results)))


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - Anemia_Class_accuracy: 0.6422 - Anemia_Class_loss: 0.6584 - Hemoglobin_loss: 6.4439 - Hemoglobin_mae: 2.0500 - loss: 7.1372
Test Results: {'loss': 7.137179374694824, 'compile_metrics': 0.6584085822105408, 'Anemia_Class_loss': 6.443858623504639, 'Hemoglobin_loss': 0.642201840877533}


In [42]:
# Make predictions
pred_class, pred_hb = final_model.predict([Conj_test, Text_test])

# Convert classification predictions to 0/1
pred_class_label = (pred_class > 0.5).astype(int)

# Print first 20 examples for quick check
print(f"{'Index':<6} {'Actual Class':<12} {'Pred Class':<12} {'Actual Hb':<10} {'Pred Hb':<10}")
for i in range(20):
    print(f"{i:<6} {y_test_class[i]:<12} {pred_class_label[i][0]:<12} {y_test_hb[i]:<10.2f} {pred_hb[i][0]:<10.2f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Index  Actual Class Pred Class   Actual Hb  Pred Hb   
0      1.0          1            12.40      10.65     
1      1.0          1            9.90       10.99     
2      1.0          1            9.90       11.48     
3      0.0          1            12.90      11.81     
4      1.0          1            8.20       11.50     
5      1.0          1            8.30       11.78     
6      1.0          1            8.30       11.57     
7      1.0          1            8.30       11.30     
8      1.0          1            8.30       10.67     
9      0.0          1            11.90      11.26     
10     0.0          1            12.70      10.80     
11     0.0          1            12.40      10.58     
12     0.0          1            11.00      10.86     
13     1.0          1            10.10      11.39     
14     1.0          1            10.70      11.77     
15     0.0          1            12.90      11.56     
16     1.0          1      